# Connections Solver


#### md segment

In [ ]:
# code segment

### LLM Few-Shot

### Instructions for running example:
    1. Get API KEY from [Cerebras Website]("https://www.cerebras.ai/"). 
    2. Save it to environment as "CEREBRAS_API_KEY".
    3. Run the next two cells

In [9]:
import os
from cerebras.cloud.sdk import Cerebras
from conn import accuracy_min_swaps, accuracy_zero_one, evaluate
from data_loader import gold_groups_from_row, get_train_test_split
import time
import src.project_notebook_imports_few_shot_LLM as fs

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

ds_train, ds_test = get_train_test_split()
# only small amout of test puzzles for showcase
ds_test = ds_test.select(range(30))
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

Training puzzles: 521
Testing puzzles: 30


In [11]:
N_EVAL = len(ds_test)

results_list = []

few_shot_values = [1, 5]
for k in few_shot_values:

    start = time.time()

    results = evaluate(
        ds_test,
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: fs.solve_puzzle(words, k=k, split_for_few_shot=ds_train, client=client),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=False,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    end = time.time()
    runtime = end - start

    results_list.append({
        "k": k,
        "zero_one_accuracy": acc,
        "mean_swaps": mean_swaps,
        "runtime": runtime,
        "n_eval": n_eval
    })

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")
    print(f"k={k} | Runtime: {runtime:.2f} seconds")


Invalid LLM output: GROUP 1: TEA || TEE (GOLF) || TEE (SHIRT) || TEE (SHIRT)
GROUP 2: COMB || GEAR || SAW || ZIPPER
GROUP 3: FUDGE || GEEZ || NUTS || RATS
GROUP 4: BANK || BED || DELTA || MOUTH
Correct Answer: ['TEA', 'TEE (GOLF)', 'TEE (SHIRT)', 'TI (MUSICAL NOTE)', 'COMB', 'GEAR', 'SAW', 'ZIPPER', 'FUDGE', 'GEEZ', 'NUTS', 'RATS', 'BANK', 'BED', 'DELTA', 'MOUTH']
Retrying (1/4)
Invalid LLM output: GROUP 1: TEA || TEE (GOLF) || TEE (SHIRT) || TEE (SHIRT)
GROUP 2: COMB || GEAR || SAW || ZIPPER
GROUP 3: FUDGE || GEEZ || NUTS || RATS
GROUP 4: BANK || BED || DELTA || MOUTH
Correct Answer: ['TEA', 'TEE (GOLF)', 'TEE (SHIRT)', 'TI (MUSICAL NOTE)', 'COMB', 'GEAR', 'SAW', 'ZIPPER', 'FUDGE', 'GEEZ', 'NUTS', 'RATS', 'BANK', 'BED', 'DELTA', 'MOUTH']
Retrying (2/4)
k=1 | Zero-one accuracy: 0.6667  (n=30, requested=30)
k=1 | Mean 1-1 swaps to correct: 0.83  (n=30)
k=1 | Runtime: 57.60 seconds
k=5 | Zero-one accuracy: 0.8333  (n=30, requested=30)
k=5 | Mean 1-1 swaps to correct: 0.33  (n=30)
k=5 | R